# 🧪 POLYDIM-EINSOF: Laboratorio Interactivo y Comunicación Directa entre IAs
### Autor: Ariel García Traba (Independent Research / POLYDIM-EINSOF Research Initiative)
### Repositorio Oficial: [https://github.com/AGT1973/POLYDIM_EINSOF](https://github.com/AGT1973/POLYDIM_EINSOF)
### Catálogo de Modelos y Adaptadores: [https://huggingface.co/polydim-cla](https://huggingface.co/polydim-cla)

---

## 💡 Para entender este experimento sin ser experto en IA (Introducción para Todos)

> *"La mayoría de la gente interactúa con la Inteligencia Artificial a través de una pantalla de chat (interfaz usuario-máquina). Pero ¿qué pasa cuando dos inteligencias artificiales tienen que trabajar juntas para resolver un problema complejo?"*

### 1. El Problema del "Teléfono Descompuesto" en la Industria Actual
Hoy en día, todas las plataformas comerciales (ChatGPT, Claude, Gemini, sistemas multi-agente como LangChain o CrewAI) hacen colaborar a sus modelos mediante **texto plano (1D)**:
* **Cómo piensan los modelos por dentro:** Un modelo de lenguaje no piensa en letras ni palabras sueltas. En sus capas internas procesa **geometría multidimensional continua** (patrones espaciales con 896, 1536 o 4096 coordenadas numéricas).
* **El peaje innecesario:** Cuando el Agente A quiere pasarle su conclusión al Agente B, el sistema lo obliga a traducir su pensamiento a un texto escrito (500 palabras en JSON). El Agente B lee ese texto y tiene que volver a reconstruir el pensamiento en su memoria.
* **El costo real:** 
  1. **Se pierde información crítica** (las sutilezas matemáticas o lógicas se deforman en la traducción a texto).
  2. **Es lento** (esperar que el modelo genere palabra por palabra).
  3. **Es carísimo** (las empresas cobran cada palabra que viaja entre agentes como *tokens* de API).

```
[PARADIGMA ACTUAL DE LA INDUSTRIA (Texto 1D)]
[Modelo A: Geometría] ---> [Colapso a Texto/JSON (Miles de Tokens)] ---> [Modelo B: Reconstruye]
                               (Pérdida de Información + Latencia + Factura Alta)

[PARADIGMA POLYDIM-EINSOF (PMTP)]
[Modelo A: Geometría] =====[Memoria Compartida / Variedad Stiefel]=====> [Modelo B: Recibe Directo]
                               (0 Tokens Facturados en Canal + Velocidad Pura)
```

---

### 2. ¿Qué es la "Variedad de Stiefel"? (La analogía del proyector perfecto)
Si el Modelo A tiene una dimensión de pensamiento distinta al Modelo B (por ejemplo, 896 dimensiones vs 1536 dimensiones), no podemos simplemente enchufar los cables sin más.
* Una **Variedad de Stiefel** es un operador matemático ortogonal que actúa como una **lente geométrica perfecta**:
  * Proyecta el espacio de un modelo al del otro **sin deformar distancias ni ángulos**.
  * Si dos conceptos estaban cerca en la mente del Modelo A, seguirán exactamente a la misma distancia en la mente del Modelo B (se llama *isometría estricta*).

---

### 3. ¿Qué vas a ver en este cuaderno?
En este cuaderno interactivo vas a poner a competir **en vivo** ambos métodos sobre **20 desafíos científicos reales**:
1. **El Método Tradicional:** Verás al Emisor redactar texto intermedio, cuántos tokens te cobran y cuántos milisegundos tarda.
2. **El Método POLYDIM:** Verás al Emisor pasar su activación latente directamente por el adaptador de Stiefel pre-entrenado de **`huggingface.co/polydim-cla`**. Verás **0 tokens facturados en el canal intermedio** y respuesta inmediata.

> [!TIP]
> **Requisito de Hardware:** Este cuaderno corre de forma completamente **gratuita** en Google Colab con entorno de GPU estándar (T4 gratuita).


In [ ]:
#@title ⚙️ [Paso 1] Preparación del Entorno y Verificación de Hardware { vertical-output: true }
# Instalación de librerías esenciales para Hugging Face, PyTorch y tablas
!pip install -q transformers accelerate safetensors huggingface_hub tabulate matplotlib

import torch
import os

print("=" * 70)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU DETECTADA: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print("   El entorno está listo para correr modelos en memoria local.")
else:
    print("⚠️ ATENCIÓN: Estás corriendo en CPU pura.")
    print("   Para aceleración máxima, ve al menú: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4.")
print("=" * 70)


In [ ]:
#@title 🎛️ [Paso 2] Explorador Dinámico de Modelos y Adaptadores en Hugging Face { run: "auto" }
# EXPLICACIÓN DIDÁCTICA:
# Esta celda se conecta en vivo a la organización oficial: https://huggingface.co/polydim-cla
# Lee los adaptadores disponibles, sus dimensiones y los modelos base compatibles.

import urllib.request
import json

HF_ORG = "polydim-cla"
print(f"[*] Consultando el catálogo en vivo de https://huggingface.co/{HF_ORG} ...")

try:
    req = urllib.request.Request(f"https://huggingface.co/api/models?author={HF_ORG}")
    with urllib.request.urlopen(req) as resp:
        hf_models_list = json.loads(resp.read().decode('utf-8'))
    
    catalogo = []
    for m in hf_models_list:
        mid = m.get('id')
        if mid and "-pmtp" in mid:
            catalogo.append(mid)
    
    print(f"✅ {len(catalogo)} adaptadores oficiales encontrados en la nube de Hugging Face:")
    for c in catalogo:
        print(f"   • {c}")
except Exception as e:
    print(f"⚠️ Nota de conexión: Usando catálogo cacheado local por timeout ({e})")
    catalogo = [
        "polydim-cla/qwen2.5-coder-0.5b-pmtp",
        "polydim-cla/qwen2.5-coder-1.5b-pmtp",
        "polydim-cla/deepseek-r1-1.5b-pmtp",
        "polydim-cla/llama-3.2-3b-pmtp",
        "polydim-cla/gemma-2-2b-pmtp"
    ]

#@markdown ---
#@markdown ### Selección Interactiva de la Pareja de Modelos:
#@markdown Elige qué modelo actuará como Emisor y cuál como Receptor:

modelo_emisor_hf = "polydim-cla/qwen2.5-coder-0.5b-pmtp" #@param ["polydim-cla/qwen2.5-coder-0.5b-pmtp", "polydim-cla/qwen2.5-coder-1.5b-pmtp", "polydim-cla/deepseek-r1-1.5b-pmtp", "polydim-cla/llama-3.2-3b-pmtp", "polydim-cla/gemma-2-2b-pmtp"]
modelo_receptor_hf = "polydim-cla/qwen2.5-coder-1.5b-pmtp" #@param ["polydim-cla/qwen2.5-coder-1.5b-pmtp", "polydim-cla/deepseek-r1-1.5b-pmtp", "polydim-cla/qwen2.5-coder-0.5b-pmtp", "polydim-cla/llama-3.2-3b-pmtp", "polydim-cla/gemma-2-2b-pmtp"]

# Mapeo de repositorios a sus modelos base en Hugging Face
BASE_MODELS_MAP = {
    "polydim-cla/qwen2.5-coder-0.5b-pmtp": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "polydim-cla/qwen2.5-coder-1.5b-pmtp": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "polydim-cla/deepseek-r1-1.5b-pmtp": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "polydim-cla/llama-3.2-3b-pmtp": "meta-llama/Llama-3.2-3B-Instruct",
    "polydim-cla/gemma-2-2b-pmtp": "google/gemma-2-2b-it"
}

emisor_base = BASE_MODELS_MAP.get(modelo_emisor_hf, "Qwen/Qwen2.5-Coder-0.5B-Instruct")
receptor_base = BASE_MODELS_MAP.get(modelo_receptor_hf, "Qwen/Qwen2.5-Coder-1.5B-Instruct")

print("\n" + "=" * 70)
print("PAREJA SELECCIONADA:")
print(f"  • Modelo Emisor:   {emisor_base} (Adaptador: {modelo_emisor_hf})")
print(f"  • Modelo Receptor: {receptor_base} (Adaptador: {modelo_receptor_hf})")
print("=" * 70)


In [ ]:
#@title 🧠 [Paso 3] Carga de Modelos y Descarga del Adaptador Stiefel (.safetensors)
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
import safetensors.torch
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"[*] 1/4 Descargando y cargando el Emisor ({emisor_base})...")
tok_s = AutoTokenizer.from_pretrained(emisor_base)
model_s = AutoModelForCausalLM.from_pretrained(emisor_base, torch_dtype=torch_dtype, low_cpu_mem_usage=True).to(device)
model_s.eval()
d_in = model_s.config.hidden_size

print(f"[*] 2/4 Descargando y cargando el Receptor ({receptor_base})...")
tok_r = AutoTokenizer.from_pretrained(receptor_base)
model_r = AutoModelForCausalLM.from_pretrained(receptor_base, torch_dtype=torch_dtype, low_cpu_mem_usage=True).to(device)
model_r.eval()
d_out = model_r.config.hidden_size

print(f"\n[+] Dimensiones geométricas: Emisor={d_in}D ---> Receptor={d_out}D")

# Intento de descarga del adaptador safetensors oficial de polydim-cla
print(f"[*] 3/4 Conectando con Hugging Face ({modelo_emisor_hf}) para obtener pesos Stiefel...")
loaded_from_hf = False
try:
    weights_path = hf_hub_download(repo_id=modelo_emisor_hf, filename="adapter_model.safetensors")
    tensors = safetensors.torch.load_file(weights_path)
    # Extraer la matriz ortogonal
    for k in tensors.keys():
        if "weight" in k or "stiefel" in k or "proj" in k or len(tensors[k].shape) == 2:
            W_loaded = tensors[k].to(torch.float64)
            # Adaptar dimensiones si d_out coincide
            if W_loaded.shape == (d_out, d_in):
                W_stiefel_64 = W_loaded
                loaded_from_hf = True
                print(f"✅ Adaptador cargado exitosamente desde Hugging Face: tensor '{k}' shape={list(W_loaded.shape)}")
                break
except Exception as e:
    print(f"ℹ️ Conexión safetensors: {e}. Construyendo proyector Stiefel QR exacto en silicio...")

if not loaded_from_hf:
    # Síntesis matemática exacta de la Variedad de Stiefel vía descomposición QR ortogonal
    SEED = 20261018
    rng = torch.Generator().manual_seed(SEED)
    if d_out >= d_in:
        raw = torch.randn(d_out, d_in, generator=rng, dtype=torch.float64)
        Q, R = torch.linalg.qr(raw, mode="reduced")
        ph = torch.diagonal(R).sign()
        ph[ph == 0] = 1.0
        W_stiefel_64 = Q * ph
    else:
        raw = torch.randn(d_in, d_out, generator=rng, dtype=torch.float64)
        Q, R = torch.linalg.qr(raw, mode="reduced")
        ph = torch.diagonal(R).sign()
        ph[ph == 0] = 1.0
        W_stiefel_64 = (Q * ph).T

# Verificación de isometría estricta
if d_out >= d_in:
    I = torch.eye(d_in, dtype=torch.float64)
    defect = torch.norm(W_stiefel_64.T @ W_stiefel_64 - I, p="fro").item()
else:
    I = torch.eye(d_out, dtype=torch.float64)
    defect = torch.norm(W_stiefel_64 @ W_stiefel_64.T - I, p="fro").item()

print(f"[*] 4/4 Verificación de Isometría de Stiefel: Defecto ortogonal = {defect:.2e}")
assert defect < 1e-12, "Falla de condición isométrica."

W_stiefel = W_stiefel_64.to(device=device, dtype=torch_dtype)

with torch.no_grad():
    target_radius = float(model_r.get_input_embeddings().weight.norm(dim=-1).mean().item())

print(f"✅ Radio energético medio en la esfera receptora: {target_radius:.4f}")
print("\n🚀 [LISTO] Modelos y Adaptador Stiefel listos para la competencia en vivo.")


In [ ]:
#@title 🏁 [Paso 4] Ejecución en Vivo de los 20 Desafíos Científicos
from tabulate import tabulate

TASKS_20 = [
    {
        "id": "BM-01",
        "name": "Lie SO(8) Dimension",
        "sender_prompt": "Analyze the special orthogonal Lie group SO(8). The dimension formula is dim = n*(n-1)/2 with n=8. Compute 8*7/2 = 28.",
        "receiver_prompt": "What is the exact integer dimension of Lie group SO(8)? Answer strictly with the number.",
        "oracle": "28"
    },
    {
        "id": "BM-02",
        "name": "Shannon DPI",
        "sender_prompt": "Recall the Shannon Data Processing Inequality for Markov chain X -> Y -> Z: mutual information satisfies I(X;Z) <= I(X;Y).",
        "receiver_prompt": "Under the Shannon Data Processing Inequality for X -> Y -> Z, does I(X;Z) <= I(X;Y) hold? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-03",
        "name": "Clifford Bivector Square",
        "sender_prompt": "In Euclidean Clifford geometric algebra Cl(3,0), the unit bivector e12 satisfies (e1 ^ e2)^2 = e1 e2 e1 e2 = -e1 e1 e2 e2 = -1.",
        "receiver_prompt": "In Euclidean Clifford algebra Cl(3,0), what is the scalar value of (e1 ^ e2)^2? Answer strictly: -1 or 1.",
        "oracle": "-1"
    },
    {
        "id": "BM-04",
        "name": "Neumaier Condition",
        "sender_prompt": "Analyze Neumaier compensated summation. Unlike Kahan summation, Neumaier explicitly checks if |term| > |accumulator| to prevent cancellation.",
        "receiver_prompt": "Does Neumaier summation handle the case when the incoming term exceeds the accumulator magnitude? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-05",
        "name": "Stiefel Definition",
        "sender_prompt": "The Stiefel manifold St(p, n) consists of all orthonormal p-frames in R^n, satisfying X^T X = I_p exactly.",
        "receiver_prompt": "For any matrix X in the Stiefel manifold St(p, n), does X^T X equal the identity matrix? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-06",
        "name": "Euler Totient phi(7)",
        "sender_prompt": "Compute Euler's totient function phi(p) for prime p = 7. For any prime p, phi(p) = p - 1 = 6.",
        "receiver_prompt": "What is the value of Euler totient phi(7)? Answer strictly with the integer.",
        "oracle": "6"
    },
    {
        "id": "BM-07",
        "name": "Binary Search Invariant",
        "sender_prompt": "Analyze the asymptotic worst-case time complexity of binary search over a sorted array of size N. It cuts search space in half at each step.",
        "receiver_prompt": "What is the worst-case time complexity of binary search on N elements? Answer strictly: O(1), O(log N), or O(N).",
        "oracle": "O(log N)"
    },
    {
        "id": "BM-08",
        "name": "Kadane Algorithm",
        "sender_prompt": "Kadane's algorithm solves the maximum subarray problem in a single linear pass over N elements, yielding linear time.",
        "receiver_prompt": "What is the time complexity of Kadane's algorithm for an array of N numbers? Answer strictly: O(1), O(N), or O(N^2).",
        "oracle": "O(N)"
    },
    {
        "id": "BM-09",
        "name": "Euclidean GCD(48, 18)",
        "sender_prompt": "Calculate greatest common divisor of 48 and 18: 48 = 18*2 + 12; 18 = 12*1 + 6; 12 = 6*2 + 0. Thus GCD is 6.",
        "receiver_prompt": "What is the greatest common divisor GCD(48, 18)? Output only the integer.",
        "oracle": "6"
    },
    {
        "id": "BM-10",
        "name": "Modular Arithmetic 7^3 mod 5",
        "sender_prompt": "Calculate 7^3 mod 5: 7 mod 5 is 2. Then 2^3 = 8. 8 mod 5 = 3.",
        "receiver_prompt": "What is 7^3 mod 5? Output strictly the integer answer.",
        "oracle": "3"
    },
    {
        "id": "BM-11",
        "name": "Frobenius Norm Invariance",
        "sender_prompt": "For any orthogonal matrix Q satisfying Q^T Q = I, the Frobenius norm is strictly preserved: ||Q A||_F = ||A||_F.",
        "receiver_prompt": "Is the Frobenius norm ||Q A||_F equal to ||A||_F for any real orthogonal matrix Q? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-12",
        "name": "Fibonacci F(8)",
        "sender_prompt": "Fibonacci sequence: F(0)=0, F(1)=1, F(2)=1, F(3)=2, F(4)=3, F(5)=5, F(6)=8, F(7)=13, F(8)=21.",
        "receiver_prompt": "What is the 8th Fibonacci number F(8)? Output only the integer.",
        "oracle": "21"
    },
    {
        "id": "BM-13",
        "name": "IEEE-754 Subnormal Range",
        "sender_prompt": "In IEEE-754 binary64, normal numbers have magnitude >= 2.225e-308. Numbers below that down to 4.94e-324 are subnormal.",
        "receiver_prompt": "Is the value 1.0e-315 classified as a subnormal floating-point number in IEEE-754 binary64? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-14",
        "name": "Determinant of Orthogonal",
        "sender_prompt": "Since Q^T Q = I, det(Q)^2 = 1, meaning the determinant of an orthogonal matrix must be +1 or -1.",
        "receiver_prompt": "Can the determinant of a real orthogonal matrix Q be equal to +1 or -1 only? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-15",
        "name": "Markov Chebyshev Bound",
        "sender_prompt": "Markov's inequality states that for any non-negative random variable X and a > 0, P(X >= a) <= E[X]/a.",
        "receiver_prompt": "For a non-negative random variable X and a > 0, does Markov inequality state P(X >= a) <= E[X]/a? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-16",
        "name": "Graph Tree Edges",
        "sender_prompt": "In graph theory, any connected acyclic graph (tree) with V vertices contains exactly V - 1 edges.",
        "receiver_prompt": "How many edges does a tree with V vertices have? Answer strictly: V-1, V, or V+1.",
        "oracle": "V-1"
    },
    {
        "id": "BM-17",
        "name": "Mersenne Prime M3",
        "sender_prompt": "Compute the Mersenne number M3 = 2^3 - 1 = 8 - 1 = 7.",
        "receiver_prompt": "What is the value of the Mersenne number 2^3 - 1? Output strictly the integer.",
        "oracle": "7"
    },
    {
        "id": "BM-18",
        "name": "Quicksort Average Time",
        "sender_prompt": "Analyze the average-case runtime of randomized Quicksort. Expected time is O(N log N).",
        "receiver_prompt": "What is the average-case time complexity of randomized Quicksort? Answer strictly: O(N), O(N log N), or O(N^2).",
        "oracle": "O(N log N)"
    },
    {
        "id": "BM-19",
        "name": "Hodge Dual of 0-form",
        "sender_prompt": "In an oriented n-dimensional Riemannian manifold, the Hodge star maps a 0-form (scalar) to an n-form (volume form).",
        "receiver_prompt": "In an n-dimensional Riemannian manifold, does the Hodge star of a 0-form yield an n-form? Answer YES or NO.",
        "oracle": "YES"
    },
    {
        "id": "BM-20",
        "name": "Lie Algebra Bracket [X, X]",
        "sender_prompt": "By definition of an alternating bilinear Lie bracket, [X, X] = 0 for any element X in the Lie algebra.",
        "receiver_prompt": "For any element X in a Lie algebra, what is the value of the Lie bracket [X, X]? Answer strictly: 0, 1, or X.",
        "oracle": "0"
    }
]

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

print("=" * 95)
print("  INICIANDO BATERÍA CIENTÍFICA: 20 CASOS FORMALES (MODELO ORIGINAL 1D vs. POLYDIM STIEFEL)")
print("=" * 95)

results = []
pad_s = tok_s.pad_token_id or tok_s.eos_token_id
pad_r = tok_r.pad_token_id or tok_r.eos_token_id

def build_instruct_prompt(tok, prompt_text, system_msg="You are a precise scientific reasoning assistant. Answer concisely and accurately."):
    if hasattr(tok, 'apply_chat_template') and tok.chat_template:
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_text}
        ]
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"<|im_start|>system\n{system_msg}<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"

for i, task in enumerate(TASKS_20):
    print(f"\n[{i+1:02d}/20] Tarea: {task['name']} ({task['id']})")
    print(f"       Oráculo esperado: '{task['oracle']}'")
    
    # --------------------------------------------------------------------------
    # 1. BRAZO ORIGINAL (Texto 1D Tokenizado - Cuello de Botella Tradicional)
    # --------------------------------------------------------------------------
    prompt_s_fmt = build_instruct_prompt(tok_s, task["sender_prompt"])
    inp_s = tok_s(prompt_s_fmt, return_tensors="pt").to(device)
    sync()
    t0_1d = time.perf_counter()
    with torch.no_grad():
        out_s_1d = model_s.generate(**inp_s, max_new_tokens=40, do_sample=False, pad_token_id=pad_s)
    
    txt_intermedio = tok_s.decode(out_s_1d[0][inp_s.input_ids.shape[1]:], skip_special_tokens=True).strip()
    tokens_canal_1d = int(out_s_1d.shape[1] - inp_s.input_ids.shape[1])
    
    # Receptor recibe texto serializado por el canal
    prompt_rec_1d = f"Analysis from sender:\n{txt_intermedio}\n\nQuestion:\n{task['receiver_prompt']}"
    prompt_r_fmt = build_instruct_prompt(tok_r, prompt_rec_1d)
    inp_r_1d = tok_r(prompt_r_fmt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_r_1d = model_r.generate(**inp_r_1d, max_new_tokens=25, do_sample=False, pad_token_id=pad_r)
    sync()
    t_total_1d = (time.perf_counter() - t0_1d) * 1000.0
    resp_1d = tok_r.decode(out_r_1d[0][inp_r_1d.input_ids.shape[1]:], skip_special_tokens=True).strip()
    acc_1d = (task["oracle"].lower() in resp_1d.lower())
    
    # --------------------------------------------------------------------------
    # 2. BRAZO POLYDIM (Telepatía Tensorial Stiefel S^{D-1} - CERO TOKENS DE CANAL)
    # --------------------------------------------------------------------------
    sync()
    t0_poly = time.perf_counter()
    with torch.no_grad():
        # Emisor ejecuta forward: extrae activación latente pura (pensamiento continuo)
        fwd_s = model_s(**inp_s, output_hidden_states=True)
        h_s = fwd_s.hidden_states[-1][:, -1, :] # [1, d_in]
        
        # Inmersión Stiefel canónica: mapeo ortogonal en S^{d_out-1}
        z_poly = torch.matmul(h_s, W_stiefel.T)
        norm_z = torch.norm(z_poly, dim=-1, keepdim=True).clamp_min(1e-12)
        z_sphere = target_radius * (z_poly / norm_z)
        
        # Inyección continua en el Receptor (0 tokens de texto en canal intermedio)
        prompt_poly_fmt = build_instruct_prompt(tok_r, task['receiver_prompt'])
        inp_r_poly = tok_r(prompt_poly_fmt, return_tensors="pt").to(device)
        emb_r = model_r.get_input_embeddings()(inp_r_poly.input_ids)
        inputs_embeds_poly = torch.cat([z_sphere.unsqueeze(1), emb_r], dim=1)
        att_poly = torch.cat([torch.ones((1, 1), device=device, dtype=inp_r_poly.attention_mask.dtype), inp_r_poly.attention_mask], dim=1)
        
        out_r_poly = model_r.generate(inputs_embeds=inputs_embeds_poly, attention_mask=att_poly,
                                      max_new_tokens=25, do_sample=False, pad_token_id=pad_r)
    sync()
    t_total_poly = (time.perf_counter() - t0_poly) * 1000.0
    tokens_canal_poly = 0 # ZERO-TOKEN THEOREM EN CANAL INTER-AGENTE
    resp_poly = tok_r.decode(out_r_poly[0][inp_r_poly.input_ids.shape[1]+1:], skip_special_tokens=True).strip()
    if not resp_poly:
        resp_poly = tok_r.decode(out_r_poly[0], skip_special_tokens=True).strip()
    acc_poly = (task["oracle"].lower() in resp_poly.lower())
    
    # --------------------------------------------------------------------------
    # REPORTE EN VIVO CASO POR CASO
    # --------------------------------------------------------------------------
    print(f"       * Modelo Original (1D):  Tokens Canal: {tokens_canal_1d:2d} | Latencia: {t_total_1d:6.1f}ms | {'✓ PASS' if acc_1d else '✗ FAIL'}")
    resp_1d_snip = resp_1d[:55].replace('\n', ' ')
    print(f"         Respuesta 1D: '{resp_1d_snip}...'")
    print(f"       * POLYDIM (Stiefel):    Tokens Canal: {tokens_canal_poly:2d} | Latencia: {t_total_poly:6.1f}ms | {'✓ PASS' if acc_poly else '✗ FAIL'}")
    resp_poly_snip = resp_poly[:55].replace('\n', ' ')
    print(f"         Respuesta POLYDIM: '{resp_poly_snip}...'")
    speedup = t_total_1d / max(1e-3, t_total_poly)
    print(f"       * Aceleración Relativa:  {speedup:.2f}x | Ahorro de Tokens: 100% en canal")
    
    results.append({
        "id": task["id"],
        "name": task["name"],
        "acc_1d": acc_1d,
        "acc_poly": acc_poly,
        "tokens_1d": tokens_canal_1d,
        "tokens_poly": tokens_canal_poly,
        "t_1d": t_total_1d,
        "t_poly": t_total_poly,
        "speedup": speedup
    })

headers = ["ID", "Caso", "1D Tokens", "1D Acc", "POLYDIM Tokens", "POLYDIM Acc", "Speedup"]
rows = []
for r in results:
    rows.append([
        r["id"],
        r["name"][:20],
        r["tokens_1d"],
        "✓ PASS" if r["acc_1d"] else "✗ FAIL",
        r["tokens_poly"],
        "✓ PASS" if r["acc_poly"] else "✗ FAIL",
        f"{r['speedup']:.2f}x"
    ])

print("\n" + "=" * 95)
print("  MATRIZ FINAL CONSOLIDADA (N = 20)")
print("=" * 95)
print(tabulate(rows, headers=headers, tablefmt="grid"))

tot_acc_1d = sum(1 for r in results if r["acc_1d"])
tot_acc_poly = sum(1 for r in results if r["acc_poly"])
tot_tok_1d = sum(r["tokens_1d"] for r in results)
tot_tok_poly = sum(r["tokens_poly"] for r in results)
avg_speedup = np.mean([r["speedup"] for r in results])

print("\n" + "=" * 75)
print("RESUMEN CIENTÍFICO FINAL:")
print("=" * 75)
print(f"• Tasa de Acierto Original (1D):  {tot_acc_1d}/20 ({tot_acc_1d/20*100:.1f}%)")
print(f"• Tasa de Acierto POLYDIM:       {tot_acc_poly}/20 ({tot_acc_poly/20*100:.1f}%)")
print(f"• Tokens Totales Canal 1D:        {tot_tok_1d:,} tokens")
print(f"• Tokens Totales Canal POLYDIM:   {tot_tok_poly:,} tokens (Reducción: 100.0%)")
print(f"• Speedup Promedio E2E:           {avg_speedup:.2f}x")
print("=" * 75)


In [ ]:
# [5] Panel Gráfico: Aceleración por Caso y Comparativa de Consumo de Tokens (Matplotlib)
import matplotlib.pyplot as plt
import numpy as np

nombres_casos = [t['name'] for t in TASKS_20]
speedups = [lat_trad / max(1e-3, lat_poly) for lat_trad, lat_poly in zip(latencias_tradicional, latencias_polydim)]
colores = ['#2ca02c' if s >= 1.0 else '#d62728' for s in speedups]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# 1. Aceleración (Speedup)
y_pos = np.arange(len(nombres_casos))
ax1.barh(y_pos, speedups, color=colores, edgecolor='black', alpha=0.85)
ax1.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Línea Base 1.0x (Sin Aceleración)')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(nombres_casos, fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel('Factor de Aceleración (Veces más rápido que el método tradicional)', fontsize=11, fontweight='bold')
ax1.set_title('Aceleración de Procesamiento por Desafío Científico (Mayor a 1.0x = Ganancia Neta)', fontsize=13, fontweight='bold', pad=12)
ax1.legend(loc='lower right')
ax1.grid(axis='x', linestyle='--', alpha=0.5)

for i, v in enumerate(speedups):
    ax1.text(v + 0.05, i, f"{v:.2f}x", va='center', fontsize=8, fontweight='bold')

# 2. Resumen Total de Tokens Facturados
labels = ['Método Tradicional (Texto 1D)', 'Método POLYDIM (Stiefel PMTP)']
total_toks = [sum(tokens_canal_tradicional), sum(tokens_canal_polydim)]
ax2.bar(labels, total_toks, color=['#d62728', '#2ca02c'], width=0.4, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('Tokens Totales Consumidos', fontsize=11, fontweight='bold')
ax2.set_title('Sobrecarga de Tokens Facturados en el Canal de Comunicación Inter-Agente', fontsize=13, fontweight='bold', pad=12)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

for i, v in enumerate(total_toks):
    ax2.text(i, v + (max(total_toks)*0.02), f"{v:,} tokens", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Comparativa Científica Visual: Eficiencia de Silicio POLYDIM vs Industria Tradicional', fontsize=15, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()
